In [8]:
import torch
from torch.utils.data import DataLoader
from utils import move_to

In [9]:
from problem_hrsp import HRSP
training_dataset = HRSP.make_dataset(size=20, num_samples=1280, distribution=None)
training_dataloader = DataLoader(training_dataset, batch_size=512, num_workers=0)
batch = next(iter(training_dataloader))
batch = move_to(batch, torch.device("cuda:0"))
input = batch

In [3]:
coords = input['source']
operation_time = input['operation_time']
ids = torch.arange(512, dtype=torch.int64, device=coords.device)[:, None]
length = torch.zeros(512, 6, dtype=torch.long, device=coords.device)
cur_time = torch.rand(512, 6, device=coords.device)
cur_coord = torch.zeros(512, 6, 2, device=coords.device)
coords.size()

torch.Size([512, 20, 2])

In [4]:
prev_task = torch.zeros(512, 6, dtype=torch.long, device=coords.device)
prev_task = move_to(prev_task, torch.device("cuda:0"))
prev_task

tensor([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        ...,
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]], device='cuda:0')

In [5]:
selected_task = torch.randint(1, 21, (512,))
selected_task = move_to(selected_task, torch.device("cuda:0"))
selected_task

tensor([ 7,  3, 20,  8, 10, 14, 18, 18,  1,  4, 10,  4,  3, 14,  4,  5,  7, 12,
        11, 10,  7,  2, 19, 18,  5,  5,  6, 17, 13, 11, 13, 20,  5, 11,  9, 10,
         1,  9, 13,  7,  8, 19, 12, 11,  7,  5,  7,  5, 16, 12, 12, 18,  9, 16,
         8, 13, 13, 11, 18,  2,  3, 18, 13,  4, 16,  9, 10, 19, 19,  5, 18,  2,
        15,  8,  3,  3,  1,  2, 15,  8,  5, 19, 18,  3, 11, 17, 16,  6, 14, 18,
        14, 14, 10,  4,  6, 14, 18,  1, 16, 10, 13,  2,  2,  4, 17,  2,  4, 13,
        12, 11,  4,  6, 12,  1, 15, 16,  6,  3, 14, 17, 12, 14, 12,  8, 20,  3,
        20,  2, 18, 10,  9,  2, 10, 10, 13,  8, 17,  9,  8,  5,  7,  5, 15,  5,
        13, 20,  9,  1,  2,  7,  2,  8,  6,  5,  3,  4, 13,  3,  7,  3,  8,  3,
        18,  9, 17,  9, 11, 19, 12,  5, 15, 20,  6,  9,  6,  5, 13,  2, 14, 15,
        20,  7,  2, 15, 20,  5, 16,  9,  9, 15, 11, 17,  8,  9,  4,  3, 19, 15,
        16,  7,  5, 20,  4,  6, 14, 14,  7, 20, 15,  1, 15,  9, 15, 10,  2, 20,
         4, 11, 18,  9, 15, 14,  3, 17, 

In [6]:
selected_robot = torch.rand(512, 6) < 0.4
selected_robot = move_to(selected_robot, torch.device("cuda:0"))
selected_robot

tensor([[False,  True, False,  True, False,  True],
        [False, False, False, False, False, False],
        [False, False, False, False, False,  True],
        ...,
        [False,  True, False, False,  True, False],
        [False,  True, False, False,  True, False],
        [False, False, False,  True,  True, False]], device='cuda:0')

In [7]:
prev_task[selected_robot] = selected_task.unsqueeze(-1).expand_as(selected_robot)[selected_robot]
prev_task

tensor([[ 0,  7,  0,  7,  0,  7],
        [ 0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0, 20],
        ...,
        [ 0,  1,  0,  0,  1,  0],
        [ 0, 19,  0,  0, 19,  0],
        [ 0,  0,  0, 20, 20,  0]], device='cuda:0')

In [8]:
selected_task_coord = coords[ids.squeeze(), selected_task - 1]
cur_coord[selected_robot] = selected_task_coord.unsqueeze(1).expand(-1, selected_robot.size(1), -1)[selected_robot]
print(coords[0])
cur_coord

tensor([[0.2166, 0.1982],
        [0.3557, 0.0294],
        [0.2115, 0.8456],
        [0.8818, 0.6300],
        [0.9194, 0.4547],
        [0.3375, 0.3251],
        [0.7941, 0.6007],
        [0.2777, 0.6351],
        [0.3487, 0.5488],
        [0.3504, 0.3144],
        [0.0328, 0.8508],
        [0.1760, 0.3317],
        [0.1808, 0.1907],
        [0.6201, 0.9074],
        [0.5788, 0.5677],
        [0.2971, 0.3303],
        [0.8764, 0.4715],
        [0.2043, 0.5682],
        [0.2340, 0.6876],
        [0.6861, 0.0510]], device='cuda:0')


tensor([[[0.2115, 0.8456],
         [0.2115, 0.8456],
         [0.2115, 0.8456],
         [0.2115, 0.8456],
         [0.0000, 0.0000],
         [0.2115, 0.8456]],

        [[0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.9843, 0.8830],
         [0.9843, 0.8830],
         [0.9843, 0.8830]],

        [[0.0000, 0.0000],
         [0.4038, 0.9408],
         [0.4038, 0.9408],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.4038, 0.9408]],

        ...,

        [[0.1086, 0.5220],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.1086, 0.5220]],

        [[0.8573, 0.5254],
         [0.8573, 0.5254],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.8573, 0.5254],
         [0.0000, 0.0000]],

        [[0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.6806, 0.5100]]], de

In [9]:
prev_coord = torch.zeros(512, 6, 2, device=coords.device)
diff =  cur_coord - prev_coord
distance = diff.norm(p=2, dim=-1)
length = length + distance
length

tensor([[0.8716, 0.8716, 0.8716, 0.8716, 0.0000, 0.8716],
        [0.0000, 0.0000, 0.0000, 1.3223, 1.3223, 1.3223],
        [0.0000, 1.0238, 1.0238, 0.0000, 0.0000, 1.0238],
        ...,
        [0.5331, 0.0000, 0.0000, 0.0000, 0.0000, 0.5331],
        [1.0055, 1.0055, 0.0000, 0.0000, 1.0055, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8505]], device='cuda:0')

In [10]:
updated_time = cur_time.clone()
travel_time = distance / 1
updated_time[selected_robot] += travel_time[selected_robot]
updated_time

tensor([[0.9734, 1.7376, 0.9944, 1.2304, 0.1809, 1.2539],
        [0.6473, 0.5289, 0.1153, 1.6129, 1.8812, 1.4261],
        [0.3791, 1.8869, 1.5471, 0.7710, 0.6090, 1.3359],
        ...,
        [0.8166, 0.7022, 0.1168, 0.3253, 0.0871, 0.6527],
        [1.2736, 1.1582, 0.1538, 0.6450, 1.0314, 0.9182],
        [0.3955, 0.8266, 0.7823, 0.0510, 0.5160, 1.6011]], device='cuda:0')

In [11]:
cur_time

tensor([[0.1017, 0.8660, 0.1227, 0.3588, 0.1809, 0.3823],
        [0.6473, 0.5289, 0.1153, 0.2906, 0.5589, 0.1038],
        [0.3791, 0.8631, 0.5233, 0.7710, 0.6090, 0.3121],
        ...,
        [0.2834, 0.7022, 0.1168, 0.3253, 0.0871, 0.1196],
        [0.2681, 0.1527, 0.1538, 0.6450, 0.0259, 0.9182],
        [0.3955, 0.8266, 0.7823, 0.0510, 0.5160, 0.7506]], device='cuda:0')

In [12]:
masked_time = updated_time.masked_fill(~selected_robot, float('-inf'))
max_time_per_batch, _ = masked_time.max(dim=1, keepdim=True)
updated_time[selected_robot] = max_time_per_batch.expand_as(updated_time)[selected_robot]
updated_time

tensor([[1.7376, 1.7376, 1.7376, 1.7376, 0.1809, 1.7376],
        [0.6473, 0.5289, 0.1153, 1.8812, 1.8812, 1.8812],
        [0.3791, 1.8869, 1.8869, 0.7710, 0.6090, 1.8869],
        ...,
        [0.8166, 0.7022, 0.1168, 0.3253, 0.0871, 0.8166],
        [1.2736, 1.2736, 0.1538, 0.6450, 1.2736, 0.9182],
        [0.3955, 0.8266, 0.7823, 0.0510, 0.5160, 1.6011]], device='cuda:0')

In [13]:
op_time = operation_time[ids.squeeze(), selected_task - 1]
updated_time += selected_robot * op_time.unsqueeze(1)
updated_time

tensor([[1.8935, 1.8935, 1.8935, 1.8935, 0.1809, 1.8935],
        [0.6473, 0.5289, 0.1153, 2.1680, 2.1680, 2.1680],
        [0.3791, 2.0092, 2.0092, 0.7710, 0.6090, 2.0092],
        ...,
        [0.9261, 0.7022, 0.1168, 0.3253, 0.0871, 0.9261],
        [1.5397, 1.5397, 0.1538, 0.6450, 1.5397, 0.9182],
        [0.3955, 0.8266, 0.7823, 0.0510, 0.5160, 1.8961]], device='cuda:0')

In [14]:
op_time

tensor([0.1559, 0.2868, 0.1223, 0.1826, 0.1891, 0.2036, 0.1627, 0.2258, 0.1563,
        0.2345, 0.1662, 0.1162, 0.1016, 0.1951, 0.2804, 0.1688, 0.1253, 0.2282,
        0.1328, 0.1926, 0.2295, 0.1733, 0.1805, 0.2419, 0.2162, 0.1101, 0.2487,
        0.2499, 0.2594, 0.2963, 0.2730, 0.1834, 0.1005, 0.2180, 0.2777, 0.1046,
        0.1142, 0.1488, 0.2904, 0.2755, 0.2237, 0.1952, 0.1588, 0.1728, 0.2655,
        0.1397, 0.1804, 0.2627, 0.1684, 0.2662, 0.2565, 0.2250, 0.1700, 0.2274,
        0.1648, 0.1229, 0.2987, 0.1106, 0.2243, 0.1902, 0.2699, 0.2769, 0.1373,
        0.2910, 0.2008, 0.2963, 0.2098, 0.1709, 0.1708, 0.2907, 0.1400, 0.1625,
        0.2357, 0.2492, 0.2269, 0.1825, 0.1366, 0.2692, 0.1058, 0.2685, 0.1261,
        0.1234, 0.2308, 0.2722, 0.1818, 0.2360, 0.1421, 0.2743, 0.2608, 0.2933,
        0.2056, 0.2080, 0.1317, 0.2363, 0.2670, 0.1513, 0.1662, 0.2937, 0.2984,
        0.1064, 0.2990, 0.2842, 0.1306, 0.2089, 0.1449, 0.2795, 0.2990, 0.2700,
        0.2542, 0.2001, 0.1432, 0.1646, 

In [15]:
visited_ = torch.zeros(
                    512, 1, 20,  # 注：这里不包括depot
                    dtype=torch.uint8, device=coords.device)
visited_

tensor([[[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]],

        ...,

        [[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]]], device='cuda:0', dtype=torch.uint8)

In [17]:
visited_ = visited_.scatter(-1, selected_task[:, None, None] - 1, 1)
visited_

tensor([[[0, 0, 1,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]],

        ...,

        [[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0]]], device='cuda:0', dtype=torch.uint8)

In [18]:
selected_task[0]

tensor(3, device='cuda:0')

In [13]:
(torch.rand(10) * 2 - 1)

tensor([ 0.2930,  0.2721,  0.6483, -0.7904,  0.8709, -0.1200,  0.5818, -0.3985,
         0.8772,  0.1107])